### 토크나이징 테스트

In [1]:
import re

# 백슬래시 1개로 시작하는 수식 텍스트
text = r"\frac{a}{b}"

# 1. 현재 적용된 패턴: r"\\[a-zA-Z]+"
pattern_double = r"\\[a-zA-Z]+"
res1 = re.findall(pattern_double, text)

print("=== 1. r'\\\\[a-zA-Z]+' 테스트 ===")
print("리스트 원형 (repr) :", res1)  # 파이썬 출력 규칙상 ['\\frac']로 보임
print("실제 문자열 출력   :", res1[0])  # 실제 화면 출력: \frac
print("문자열 길이 (len)  :", len(res1[0]))  # 5글자 (\, f, r, a, c)

# 2. 백슬래시를 1개만 쓴 패턴: r"\[a-zA-Z]+"
pattern_single = r"\[a-zA-Z]+"
res2_math = re.findall(pattern_single, text)
res2_bracket = re.findall(pattern_single, "[test] value")

print("\n=== 2. r'\[a-zA-Z]+' 테스트 ===")
print(r"수식 '\frac' 매칭 결과 :", res2_math)  # 매칭 실패: []
print(r"대괄호 '[test]' 매칭 결과:", res2_bracket)  # 대괄호 매칭: ['[test']

=== 1. r'\\[a-zA-Z]+' 테스트 ===
리스트 원형 (repr) : ['\\frac']
실제 문자열 출력   : \frac
문자열 길이 (len)  : 5

=== 2. r'\[a-zA-Z]+' 테스트 ===
수식 '\frac' 매칭 결과 : []
대괄호 '[test]' 매칭 결과: []


In [3]:
import re
import unittest

PATTERN = r"\\[a-zA-Z]+|[a-zA-Z0-9]+|[^\s\w]"
SAMPLE_TEXT = r"Here is \frac{a}{b} and \alpha + \beta = \gamma and x^2 \le 10"


def tokenize(text: str, pattern: str = PATTERN) -> list[str]:
    tokens = re.findall(pattern, text)
    return [t if t.startswith("\\") else t.lower() for t in tokens]


class TestLatexTokenization(unittest.TestCase):

    def test_token_classification(self):
        tokens = tokenize(SAMPLE_TEXT)

        # 전체 토큰 분해 결과
        expected_tokens = [
            "here", "is", r"\frac", "{", "a", "}", "{", "b", "}",
            "and", r"\alpha", "+", r"\beta", "=", r"\gamma",
            "and", "x", "^", "2", r"\le", "10",
        ]
        self.assertEqual(tokens, expected_tokens)

        # 1. LaTeX 명령어 분리 보존 검증
        latex_commands = [t for t in tokens if t.startswith("\\")]
        self.assertEqual(latex_commands, [r"\frac", r"\alpha", r"\beta", r"\gamma", r"\le"])

        # 2. 연산자 및 괄호 기호 분리 검증
        symbols = [t for t in tokens if re.match(r"[^\s\w]", t)]
        self.assertEqual(symbols, ["{", "}", "{", "}", "+", "=", "^"])

        # 3. 영숫자 및 변수/상수 분리 검증
        words_and_numbers = [t for t in tokens if re.match(r"^[a-zA-Z0-9]+$", t)]
        self.assertEqual(
            words_and_numbers,
            ["here", "is", "a", "b", "and", "and", "x", "2", "10"],
        )


if __name__ == "__main__":
    # 주피터 노트북 환경 호환 실행 (argv와 exit 옵션 추가)
    unittest.main(argv=["first-arg-is-ignored"], exit=False)

F
FAIL: test_token_classification (__main__.TestLatexTokenization.test_token_classification)
----------------------------------------------------------------------
Traceback (most recent call last):
  File "C:\Users\baoro\AppData\Local\Temp\ipykernel_28764\1590034555.py", line 32, in test_token_classification
    self.assertEqual(symbols, ["{", "}", "{", "}", "+", "=", "^"])
AssertionError: Lists differ: ['\\frac', '{', '}', '{', '}', '\\alpha', '[37 chars]\le'] != ['{', '}', '{', '}', '+', '=', '^']

First differing element 0:
'\\frac'
'{'

First list contains 5 additional elements.
First extra element 7:
'\\beta'

+ ['{', '}', '{', '}', '+', '=', '^']
- ['\\frac',
-  '{',
-  '}',
-  '{',
-  '}',
-  '\\alpha',
-  '+',
-  '\\beta',
-  '=',
-  '\\gamma',
-  '^',
-  '\\le']

----------------------------------------------------------------------
Ran 1 test in 0.002s

FAILED (failures=1)


In [2]:
import re
import unittest

PATTERN = r"\\[a-zA-Z]+|[a-zA-Z0-9]+|[^\s\w]"
SAMPLE_TEXT = r"Here is \frac{a}{b} and \alpha + \beta = \gamma and x^2 \le 10"


def tokenize(text: str, pattern: str = PATTERN) -> list[str]:
    tokens = re.findall(pattern, text)
    return [t if t.startswith("\\") else t.lower() for t in tokens]

tokens = tokenize(SAMPLE_TEXT)
print(tokens)

['here', 'is', '\\frac', '{', 'a', '}', '{', 'b', '}', 'and', '\\alpha', '+', '\\beta', '=', '\\gamma', 'and', 'x', '^', '2', '\\le', '10']


### ollama 모델 리스트 확인 테스트

In [1]:
import ollama

try:
    # 1. Ollama 클라이언트 초기화 (기본 주소: http://localhost:11434)
    client = ollama.Client()
    response = client.list()

    # 2. SDK 버전 호환성(객체/딕셔너리)을 고려한 모델명 추출
    if isinstance(response, dict):
        raw_models = response.get("models", [])
        models = [
            m.get("model")
            for m in raw_models
            if isinstance(m, dict) and m.get("model")
        ]
    else:
        raw_models = getattr(response, "models", []) or []
        models = [m.model for m in raw_models if getattr(m, "model", None)]

    # 3. 결과 출력
    print(f"총 {len(models)}개의 모델 감지됨\n" + "-" * 30)
    for idx, name in enumerate(models, 1):
        print(f"{idx}. {name}")

except Exception as e:
    print(f"[ERROR] Ollama 연결 실패: {e}")
    print("-> Ollama 데스크톱 앱이 켜져 있거나 터미널에서 'ollama serve'가 구동 중인지 확인하세요.")

총 2개의 모델 감지됨
------------------------------
1. nomic-embed-text:latest
2. qwen2.5-coder:7b


### Regualr Expression matching test

In [2]:
import re

# 연(4자리), 월(2자리), 일(2자리)을 각각 소괄호 ()로 묶음
pattern = re.compile(r"(\d{4})-(\d{2})-(\d{2})")

m = pattern.match("2026-10-05")

if m:
    print(m.group(0))  # '2026-10-05' (전체 매칭 결과, .group()과 동일)
    print(m.group(1))  # '2026'       (첫 번째 괄호)
    print(m.group(2))  # '10'         (두 번째 괄호)
    print(m.group(3))  # '05'         (세 번째 괄호)

2026-10-05
2026
10
05
